# Comparación de modelos

Este notebook **no entrena nada**: solo carga las métricas exportadas por cada notebook de modelado y las concatena en una sola tabla.

Cada notebook fuente guarda su CSV en `Modelado/artifacts/`:

| Notebook fuente | Archivo de métricas |
|---|---|
| `JoinXcodex/Join.ipynb` (BLOQUE 10) | `metricas_baseline_sgd.csv` |
| `Modelado/modeladoRandomForest.ipynb` | `metricas_rf.csv` |
| `Modelado/modeladoHistGradientBoosting.ipynb` | `metricas_hgb.csv` |

Si falta alguno, ejecuta el notebook correspondiente hasta la celda de exportación de métricas.

In [6]:
import pandas as pd
import numpy as np
from pathlib import Path

ARTIFACTS_DIR = Path("artifacts")
print("Carpeta:", ARTIFACTS_DIR.resolve())
print("Archivos disponibles:")
for p in sorted(ARTIFACTS_DIR.glob("metricas_*.csv")):
    print("  -", p.name)

Carpeta: C:\Users\camil\OneDrive\Documentos\Aprendizaje de Maquina\Taller\Modelado\artifacts
Archivos disponibles:
  - metricas_baseline_sgd.csv
  - metricas_hgb.csv
  - metricas_rf.csv


In [7]:
archivos_metricas = {
    "baseline_sgd": ARTIFACTS_DIR / "metricas_baseline_sgd.csv",
    "random_forest": ARTIFACTS_DIR / "metricas_rf.csv",
    "hgb": ARTIFACTS_DIR / "metricas_hgb.csv",
}

dfs = []
for nombre, ruta in archivos_metricas.items():
    if ruta.exists():
        df = pd.read_csv(ruta)
        df["fuente"] = nombre
        dfs.append(df)
        print(f"[OK] {nombre}: {len(df)} fila(s)")
    else:
        print(f"[FALTA] {nombre} -> {ruta}")

if not dfs:
    raise FileNotFoundError(
        "No se encontraron CSV de métricas. Ejecuta antes la celda final de cada "
        "notebook de modelado para exportar los CSV."
    )

[OK] baseline_sgd: 2 fila(s)
[OK] random_forest: 1 fila(s)
[OK] hgb: 1 fila(s)


In [8]:
tabla_comparativa_modelos = pd.concat(dfs, ignore_index=True)

orden_cols = [
    "modelo", "umbral",
    "accuracy", "precision", "recall", "f1",
    "pr_auc", "roc_auc",
    "tp", "fp", "fn", "tn",
    "fuente",
]
orden_cols = [c for c in orden_cols if c in tabla_comparativa_modelos.columns]
tabla_comparativa_modelos = tabla_comparativa_modelos[orden_cols]

tabla_comparativa_modelos = tabla_comparativa_modelos.sort_values(
    "f1", ascending=False
).reset_index(drop=True)

display(tabla_comparativa_modelos.round(4))

,modelo,umbral,accuracy,precision,recall,f1,pr_auc,roc_auc,tp,fp,fn,tn,fuente
0,HistGradientBoosting tuned,0.85,0.9585,0.1216,0.2359,0.1605,0.0835,0.8114,5783,41779,18734,1392116,hgb
1,Random Forest tuned,0.8,0.9467,0.1062,0.2926,0.1558,0.0806,0.8087,7173,60399,17344,1373496,random_forest
2,SGD balanceado,0.8,0.9402,0.0928,0.2917,0.1409,0.0723,0.7768,7152,69880,17365,1364015,baseline_sgd
3,Baseline siempre 0,-,0.9832,0.0000,0.0000,0.0000,0.0168,NaN,0,0,24517,1433895,baseline_sgd


In [4]:
ruta_tabla = ARTIFACTS_DIR / "tabla_comparativa_modelos.csv"
tabla_comparativa_modelos.to_csv(ruta_tabla, index=False)
print(f"Tabla comparativa guardada en: {ruta_tabla.resolve()}")

Tabla comparativa guardada en: C:\Users\camil\OneDrive\Documentos\Aprendizaje de Maquina\Taller\Modelado\artifacts\tabla_comparativa_modelos.csv


In [5]:
# =========================================================
# SELECCIÓN DEL MODELO FINAL
# =========================================================

# Ordenar por F1-score de mayor a menor
tabla_comparativa_modelos = tabla_comparativa_modelos.sort_values(
    "f1",
    ascending=False
).reset_index(drop=True)

modelo_final = tabla_comparativa_modelos.iloc[0]

print("="*70)
print("MODELO FINAL SELECCIONADO")
print("="*70)

print(f"Modelo seleccionado: {modelo_final['modelo']}")
print(f"Umbral final: {modelo_final['umbral']}")
print(f"Accuracy: {modelo_final['accuracy']:.4f}")
print(f"Precision: {modelo_final['precision']:.4f}")
print(f"Recall: {modelo_final['recall']:.4f}")
print(f"F1-score: {modelo_final['f1']:.4f}")
print(f"PR-AUC: {modelo_final['pr_auc']:.4f}")
print(f"ROC-AUC: {modelo_final['roc_auc']:.4f}")

display(modelo_final.to_frame().T)

MODELO FINAL SELECCIONADO
Modelo seleccionado: HistGradientBoosting balanceado
Umbral final: 0.85
Accuracy: 0.9583
Precision: 0.1207
Recall: 0.2353
F1-score: 0.1596
PR-AUC: 0.0834
ROC-AUC: 0.8109


,modelo,umbral,accuracy,precision,recall,f1,pr_auc,roc_auc,tp,fp,fn,tn,fuente
0,HistGradientBoosting balanceado,0.85,0.958326,0.120709,0.235347,0.159573,0.08339,0.810912,5770,42031,18747,1391864,hgb


Se seleccionó como modelo final el **HistGradientBoosting balanceado** con umbral de decisión **0.85**, ya que presentó el mejor desempeño global entre los modelos evaluados. Aunque su recall es menor que el de otros modelos, obtuvo el mayor F1-score, PR-AUC y ROC-AUC, lo que indica un mejor equilibrio entre detección de accidentes y control de falsas alarmas.

El baseline ingenuo obtuvo una accuracy alta porque predice siempre la clase mayoritaria, pero no detecta ningún accidente. Por esta razón, no se usó la accuracy como métrica principal. En cambio, se priorizaron métricas más adecuadas para datos desbalanceados, como precision, recall, F1-score y PR-AUC.